# Kaggle PaddleOCR PP-StructureV3 Worker

Outbound WebSocket OCR worker for the gateway `/v1/ocr` endpoint.


In [ ]:
!python -m pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!python -m pip install -q websockets httpx pillow langchain-text-splitters langchain-community "paddlex[ocr]" paddleocr
!python -m pip uninstall -y torch torchvision torchaudio torchtext >/dev/null 2>&1 || true


In [ ]:
import asyncio
import base64
import json
import mimetypes
import os
import tempfile
import time
import uuid
from pathlib import Path
from urllib.parse import urlsplit

import httpx
import websockets
from paddleocr import PPStructureV3

EMBEDDED_RUNTIME_CONFIG = {}

def config_value(name, default=None):
    if name in os.environ and os.environ[name] != '':
        return os.environ[name]
    if name in EMBEDDED_RUNTIME_CONFIG and EMBEDDED_RUNTIME_CONFIG[name] != '':
        return EMBEDDED_RUNTIME_CONFIG[name]
    path = Path('worker_config.json')
    if path.exists():
        try:
            value = json.loads(path.read_text()).get(name)
            if value not in (None, ''):
                return value
        except Exception:
            pass
    return default

GATEWAY_WS_URL = str(config_value('GATEWAY_WS_URL', '')).strip()
WORKER_TOKEN = str(config_value('WORKER_TOKEN', '')).strip()
OWNER = str(config_value('OWNER', 'unknown')).strip() or 'unknown'
OCR_MODEL = str(config_value('OCR_MODEL', 'paddleocr-ppstructurev3')).strip()
OCR_CAPACITY = max(1, int(config_value('OCR_CAPACITY', 1)))
KEEPALIVE_LOG_SECONDS = int(config_value('KEEPALIVE_LOG_SECONDS', 60))
KAGGLE_ACCELERATOR = str(config_value('KAGGLE_ACCELERATOR', 'NvidiaTeslaT4'))
PADDLEOCR_DEVICE = str(config_value('PADDLEOCR_DEVICE', 'auto')).strip().lower()
PADDLEOCR_LANG = str(config_value('PADDLEOCR_LANG', '')).strip() or None

try:
    import paddle
    GPU_COUNT = paddle.device.cuda.device_count()
    GPU_NAMES = [paddle.device.cuda.get_device_name(i) for i in range(GPU_COUNT)]
except Exception as exc:
    print({'event': 'paddle_gpu_detection_failed', 'error': repr(exc)}, flush=True)
    GPU_COUNT = 0
    GPU_NAMES = []

if PADDLEOCR_DEVICE == 'auto':
    PADDLEOCR_DEVICE = 'gpu:0,1' if GPU_COUNT >= 2 else ('gpu:0' if GPU_COUNT == 1 else 'cpu')

NODE_ID = f"ocr-paddle-{OWNER}-{uuid.uuid4().hex[:8]}"
print({
    'node_id': NODE_ID,
    'owner': OWNER,
    'model': OCR_MODEL,
    'device': PADDLEOCR_DEVICE,
    'gpu_names': GPU_NAMES,
    'gateway': urlsplit(GATEWAY_WS_URL).netloc,
}, flush=True)

pipeline_kwargs = {'device': PADDLEOCR_DEVICE}
if PADDLEOCR_LANG:
    pipeline_kwargs['lang'] = PADDLEOCR_LANG
pipeline = PPStructureV3(**pipeline_kwargs)
print('PP-StructureV3 pipeline loaded', flush=True)

def ws_url_with_token():
    if not WORKER_TOKEN:
        return GATEWAY_WS_URL
    sep = '&' if '?' in GATEWAY_WS_URL else '?'
    return f'{GATEWAY_WS_URL}{sep}token={WORKER_TOKEN}'

def safe_suffix(filename, mime_type, default='.png'):
    if filename:
        suffix = Path(filename).suffix
        if suffix:
            return suffix[:12]
    if mime_type:
        guessed = mimetypes.guess_extension(mime_type.split(';')[0].strip())
        if guessed:
            return guessed
    return default

def decode_data_url(value):
    header, encoded = value.split(',', 1)
    mime_type = 'application/octet-stream'
    if header.startswith('data:'):
        mime_type = header[5:].split(';', 1)[0] or mime_type
    return base64.b64decode(encoded), mime_type

async def materialize_input(request, job_id):
    source = request.get('image_url') or request.get('document_url')
    encoded = request.get('image_base64') or request.get('document_base64')
    filename = request.get('filename') or f'{job_id}.png'
    mime_type = request.get('mime_type') or ''
    if source:
        if source.startswith('data:'):
            content, detected_mime = decode_data_url(source)
            mime_type = mime_type or detected_mime
        else:
            async with httpx.AsyncClient(timeout=120) as client:
                response = await client.get(source)
                response.raise_for_status()
                content = response.content
                mime_type = mime_type or response.headers.get('content-type', '')
                path_name = Path(urlsplit(source).path).name
                filename = request.get('filename') or path_name or filename
    elif encoded:
        if encoded.startswith('data:'):
            content, detected_mime = decode_data_url(encoded)
            mime_type = mime_type or detected_mime
        else:
            content = base64.b64decode(encoded)
    else:
        raise ValueError('OCR request has no input')
    suffix = safe_suffix(filename, mime_type)
    path = Path(tempfile.gettempdir()) / f'{job_id}{suffix}'
    path.write_bytes(content)
    return path, {'filename': filename, 'mime_type': mime_type, 'bytes': len(content)}

def to_jsonable(value, depth=0):
    if depth > 8:
        return str(type(value).__name__)
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): to_jsonable(v, depth + 1) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v, depth + 1) for v in value]
    if hasattr(value, 'tolist'):
        return to_jsonable(value.tolist(), depth + 1)
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)

def markdown_text_from_info(md_info):
    if not isinstance(md_info, dict):
        return ''
    for key in ('markdown_text', 'markdown_texts', 'markdown', 'text', 'content'):
        value = md_info.get(key)
        if isinstance(value, str) and value.strip():
            return value
        if isinstance(value, list):
            return '\n'.join(str(item) for item in value if item)
    return ''

def run_paddle_ocr(path, request, input_meta):
    started = time.time()
    output = pipeline.predict(input=str(path))
    pages = []
    markdown_list = []
    text_parts = []
    for index, res in enumerate(output):
        raw_json = to_jsonable(getattr(res, 'json', {}))
        md_info = getattr(res, 'markdown', {}) or {}
        if isinstance(md_info, dict):
            markdown_list.append(md_info)
        page_markdown = markdown_text_from_info(md_info)
        if page_markdown:
            text_parts.append(page_markdown)
        pages.append({
            'index': index,
            'markdown': page_markdown,
            'json': raw_json,
        })
    markdown = ''
    if markdown_list:
        try:
            merged = pipeline.concatenate_markdown_pages(markdown_list)
            markdown = merged[0] if isinstance(merged, tuple) else str(merged)
        except Exception:
            markdown = '\n\n'.join(text_parts)
    else:
        markdown = '\n\n'.join(text_parts)
    return {
        'text': markdown,
        'markdown': markdown,
        'pages': pages,
        'data': {'pages': pages},
        'metadata': {
            'backend': 'paddleocr-ppstructurev3',
            'device': PADDLEOCR_DEVICE,
            'input': input_meta,
            'elapsed_seconds': round(time.time() - started, 3),
        },
    }

async def handle_ocr_job(ws, job):
    job_id = str(job.get('job_id') or '')
    request = job.get('request') or {}
    try:
        path, input_meta = await materialize_input(request, job_id)
        result = await asyncio.to_thread(run_paddle_ocr, path, request, input_meta)
        await ws.send(json.dumps({'type': 'ocr_done', 'job_id': job_id, 'result': result}))
    except Exception as exc:
        await ws.send(json.dumps({'type': 'ocr_error', 'job_id': job_id, 'error': repr(exc)}))

async def heartbeat_loop(ws):
    while True:
        await asyncio.sleep(20)
        await ws.send(json.dumps({'type': 'heartbeat', 'node_id': NODE_ID}))

async def worker_loop():
    if not GATEWAY_WS_URL:
        raise RuntimeError('GATEWAY_WS_URL is required')
    while True:
        try:
            async with websockets.connect(ws_url_with_token(), ping_interval=20, ping_timeout=20, max_size=None) as ws:
                await ws.send(json.dumps({
                    'type': 'register',
                    'node_id': NODE_ID,
                    'owner': OWNER,
                    'model': OCR_MODEL,
                    'accelerator': f'{KAGGLE_ACCELERATOR}x{GPU_COUNT or 0}',
                    'capacity': OCR_CAPACITY,
                }))
                print('registered', await ws.recv(), flush=True)
                hb_task = asyncio.create_task(heartbeat_loop(ws))
                try:
                    async for raw in ws:
                        message = json.loads(raw)
                        if message.get('type') == 'terminate':
                            print({'event': 'terminate', 'reason': message.get('reason')}, flush=True)
                            return
                        if message.get('type') == 'ocr_job':
                            await handle_ocr_job(ws, message)
                finally:
                    hb_task.cancel()
        except Exception as exc:
            print('worker reconnect after error', repr(exc), flush=True)
            await asyncio.sleep(10)

await worker_loop()
